In [ ]:
# Practical 2: TF-IDF and Word2Vec

In [12]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\omcho\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [13]:
# dataset
df = pd.read_csv(r"D:\clg\GS Moze\4th year\8th sem\Practical\BE_NLP_72311560F_OM\PR_2\Dataset\sample.csv")

# showing first few rows
df.head()

,Make,Model,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Market Category,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP
0,BMW,1 Series M,2011,premium unleaded (required),335.0,6.0,MANUAL,rear wheel drive,2.0,"Factory Tuner,Luxury,High-Performance",Compact,Coupe,26,19,3916,46135
1,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Convertible,28,19,3916,40650
2,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,High-Performance",Compact,Coupe,28,20,3916,36350
3,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Coupe,28,18,3916,29450
4,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,Luxury,Compact,Convertible,28,18,3916,34500


In [ ]:
# creating text column 
df["text"] = (
    df["Make"].astype(str) + " " +
    df["Model"].astype(str) + " " +
    df["Engine Fuel Type"].astype(str) + " " +
    df["Vehicle Style"].astype(str)
)

corpus = df["text"].tolist()

print(corpus[:5])

['BMW 1 Series M premium unleaded (required) Coupe', 'BMW 1 Series premium unleaded (required) Convertible', 'BMW 1 Series premium unleaded (required) Coupe', 'BMW 1 Series premium unleaded (required) Coupe', 'BMW 1 Series premium unleaded (required) Convertible']


In [ ]:
# TF-IDF
tfidf = TfidfVectorizer()

tfidf_matrix = tfidf.fit_transform(corpus)

print("TF-IDF Features:")
print(tfidf.get_feature_names_out()[:20])   # limit for clean output

print("\nTF-IDF Matrix (first 5 rows):")
print(tfidf_matrix.toarray()[:5])

TF-IDF Features:
['10' '100' '124' '12c' '15' '150' '1500' '1500hd' '16' '190' '200' '200h'
 '200sx' '200t' '240' '240sx' '250' '2500' '250h' '2dr']

TF-IDF Matrix (first 5 rows):
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [ ]:
# Tokenization
tokenized = [word_tokenize(sent.lower()) for sent in corpus]

print(tokenized[:2])

[['bmw', '1', 'series', 'm', 'premium', 'unleaded', '(', 'required', ')', 'coupe'], ['bmw', '1', 'series', 'premium', 'unleaded', '(', 'required', ')', 'convertible']]


In [ ]:
# Word2Vec
model = Word2Vec(
    sentences=tokenized,
    vector_size=50,
    window=2,
    min_count=1
)

print("Vector for word 'bmw':")
print(model.wv['bmw'])

Vector for word 'bmw':
[ 0.18587425 -0.688927   -0.5368571   0.5692524   0.84646183  0.4641004
  0.03607557  0.4793586  -1.1562258  -0.57737106 -0.50252175  0.32919976
 -0.46676922  0.0919231   0.49942985  0.02525867  0.30117652 -0.13806026
  0.8587792  -0.3446606   0.17777792 -0.77377254  0.44299054 -0.20520794
  0.308564    0.24961631 -0.615872    0.08831707 -0.3218647   0.32569215
  0.41256458  0.13954704 -0.25014943 -0.11045817 -0.45715702 -0.352269
 -0.13725795  0.59848505  0.4085639   0.2703347  -0.36757413 -0.06938279
  0.3487893   0.9856093  -0.12538944  0.1628385   0.00168443 -0.15574108
 -0.07583047 -0.84789276]


In [ ]:
# find similar words
("Words similar to 'bmw':")
print(model.wv.most_similar("bmw"))

Words similar to 'bmw':
[('1', 0.9705244302749634), ('4', 0.9662541747093201), ('6', 0.9612263441085815), ('gran', 0.9527409672737122), ('5', 0.9438195824623108), ('7', 0.9374791383743286), ('miata', 0.9301488399505615), ('z4', 0.9293410778045654), ('x6', 0.9249213933944702), ('2', 0.9212660789489746)]


In [22]:
# Sentence Embedding
import numpy as np

def sentence_vector(sentence):
    vectors = [model.wv[w] for w in sentence if w in model.wv]
    return np.mean(vectors, axis=0)

embedding = sentence_vector(tokenized[0])

print("Sentence embedding shape:", embedding.shape)
print("Sentence embedding (first 5 values):")
print(embedding[:5])

Sentence embedding shape: (50,)
Sentence embedding (first 5 values):
[ 0.02023333 -0.30063757 -0.1204571   0.30698183  0.5024427 ]
